# Diabetic Retinopathy Screening Agent
Vicheda Narith, Maanvi Sarwadi

## Setup & Dependencies

Run the following script to load packages and dependencies

`run script.sh`

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import requests
import os

In [2]:
UNCERTAINTY_THRESHOLD = 0.5
DR_GRADES = {
    0: "No apparent DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR"
}

## Load Pre-trained Models

In [3]:
# load EyePACS straight from huggingface
from datasets import load_dataset

eyepacs = load_dataset("ctmedtech/EYEPACS", split="train")
print(eyepacs)

README.md:   0%|          | 0.00/4.40k [00:00<?, ?B/s]

rm_images/Merged_Fundus_Images_with_Capt(…):   0%|          | 0.00/407k [00:00<?, ?B/s]

rm_images/left_preprocess_sample.jpg:   0%|          | 0.00/240k [00:00<?, ?B/s]

rm_images/right_preprocess_sample.jpg:   0%|          | 0.00/223k [00:00<?, ?B/s]

EYEPACS.zip:   0%|          | 0.00/6.48G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['image'],
    num_rows: 35111
})


In [4]:
# transforms to apply images
def get_transforms():
    return transforms.Compose(
        [
            transforms.Resize((512, 512)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

# dataset
class DRDataset(Dataset):
    def __init__(self, data_source, img_dir=None, transform=None):
        self.data_source = data_source
        self.img_dir = img_dir
        self.transform = transform or get_transforms()

    def __len__(self):
        return len(self.data_source)
    
    def __getitem__(self, idx):
        row = self.data_source[idx]
        if isinstance(row, pd.Series):
            row = row.to_dict()

        if isinstance(row, dict) and 'image' in row:
            img = row['image']
        else:
            img_name = row['image_name']
            img_path = os.path.join(self.img_dir, img_name)
            img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = row['label']
        return img, label

# create the dataset and load
dataset = DRDataset(eyepacs)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
print(eyepacs.features)
print(eyepacs[0].keys())

{'image': Image(mode=None, decode=True)}
dict_keys(['image'])


In [ ]:
from datasets import load_dataset
from torch.utils.data import random_split

# Load APTOS with labels — already 224x224, no preprocessing needed
aptos = load_dataset("bumbledeep/aptos", split="train")

print(aptos)
print(aptos.features)
print(aptos[0].keys())  # should show: image, label_code, label

README.md:   0%|          | 0.00/5.36k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3662 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'label_code', 'label'],
    num_rows: 3662
})
{'image': Image(mode=None, decode=True), 'label_code': Value('int64'), 'label': Value('string')}
dict_keys(['image', 'label_code', 'label'])


In [14]:
class DRDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data      = data
        self.transform = transform or get_transforms()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data[idx]
        img   = row['image'].convert('RGB')
        label = row['label_code']  # integer 0–4
        if self.transform:
            img = self.transform(img)
        return img, label


# Train/val split (85/15)
n       = len(aptos)
n_train = int(0.85 * n)
n_val   = n - n_train

train_data, val_data = random_split(aptos, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(DRDataset(train_data), batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(DRDataset(val_data),   batch_size=32, shuffle=False, num_workers=0)

print(f'Train: {n_train} | Val: {n_val}')

Train: 3112 | Val: 550


In [ ]:
import timm

class DRVisionModel(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            num_classes=0  # remove default head
        )
        feature_dim = self.backbone.num_features  # 1536

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DRVisionModel().to(DEVICE)
print(f'Device: {DEVICE}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Device: cpu
Parameters: 11,485,741


In [ ]:
from tqdm import tqdm

def train(model, train_loader, val_loader, epochs=10):
    # Class weights to handle imbalance
    class_weights = torch.tensor([0.5, 1.5, 2.0, 2.5, 3.0]).to(DEVICE)
    criterion  = nn.CrossEntropyLoss(weight=class_weights)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1} [train]'):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc=f'Epoch {epoch+1} [val]'):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                val_loss += criterion(model(imgs), labels).item()

        train_loss /= len(train_loader)
        val_loss   /= len(val_loader)
        scheduler.step()

        print(f'Epoch {epoch+1}: train={train_loss:.4f}  val={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'checkpoints/dr_best.pth')
            print('  → Checkpoint saved')

    print('Training complete.')

In [17]:
import os
os.makedirs('checkpoints', exist_ok=True)

train(model, train_loader, val_loader, epochs=10)

Epoch 1 [train]:   0%|          | 0/98 [02:46<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Monte Carlo Dropout Inference

def mc_dropout_predict(model, image_tensor, n_passes=30):
    """
    Run n_passes stochastic forward passes with dropout active.
    Returns grade prediction + uncertainty scores.
    """
    model.eval()
    model.enable_mc_dropout()  # keep dropout on during inference

    image_tensor = image_tensor.to(DEVICE)
    all_probs = []

    with torch.no_grad():
        for _ in range(n_passes):
            logits = model(image_tensor)           
            probs  = F.softmax(logits, dim=1)      
            all_probs.append(probs.cpu().numpy())

    all_probs  = np.array(all_probs).squeeze()    
    probs_mean = all_probs.mean(axis=0)           
    all_preds  = np.argmax(all_probs, axis=1)     

    return {
        'grade':      int(np.argmax(probs_mean)),
        'grade_label': DR_GRADES[int(np.argmax(probs_mean))],
        'confidence': float(probs_mean.max()),
        'variance':   float(np.var(all_preds)),
        'entropy':    float(-np.sum(probs_mean * np.log(probs_mean + 1e-8))),
        'probs_mean': probs_mean.tolist(),
        'all_preds':  all_preds.tolist()
    }

print('MC Dropout defined.')

In [ ]:
# Validate Image
import cv2

def check_image_quality(img: Image.Image) -> dict:
    """
    Checks blur and field-of-view before grading.
    Rejects images that are too blurry or poorly framed.
    """
    img_gray = np.array(img.convert('L'))

    # Blur detection via Laplacian variance
    blur_score = cv2.Laplacian(img_gray, cv2.CV_64F).var()
    if blur_score < 50:
        return {'valid': False, 'reason': f'Image too blurry (score: {blur_score:.1f})'}

    # Field of view: retinal disc should fill >15% of frame
    non_black = np.sum(img_gray > 10) / img_gray.size
    if non_black < 0.15:
        return {'valid': False, 'reason': f'Insufficient field of view ({non_black:.1%})'}

    return {'valid': True, 'reason': 'OK'}

print('Image validation defined.')

In [ ]:
import faiss
import anthropic

# Clinical knowledge base
KNOWLEDGE_BASE = [
    {
        'id': 'aao_0',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 0 (No DR): Annual dilated eye exam. No treatment required. Optimize glycemic control.'
    },
    {
        'id': 'aao_1',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 1 (Mild NPDR): Microaneurysms only. Follow-up in 12 months. No ocular treatment needed.'
    },
    {
        'id': 'aao_2',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 2 (Moderate NPDR): Refer to ophthalmologist within 3-6 months. Consider anti-VEGF if macular edema present.'
    },
    {
        'id': 'aao_3',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 3 (Severe NPDR): Urgent referral within 1 month. High risk of progression to PDR. Panretinal photocoagulation should be considered.'
    },
    {
        'id': 'aao_4',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 4 (Proliferative DR): Emergency referral to vitreoretinal surgeon. Immediate treatment indicated. Risk of irreversible vision loss.'
    },
    {
        'id': 'pub_1',
        'source': 'Gulshan et al. JAMA 2016',
        'text': 'Deep learning achieved AUC 0.991 for referable DR detection. Sensitivity 97.5%, specificity 93.4% on EyePACS validation set.'
    },
    {
        'id': 'pub_2',
        'source': 'Nature Medicine 2022',
        'text': 'Calibrated uncertainty estimates are critical for safe AI in clinical settings. Monte Carlo Dropout is a validated approach for uncertainty quantification in medical imaging.'
    }
]

def build_faiss_index(knowledge_base, dim=128):
    """Build FAISS index from knowledge base using simple character embeddings."""
    def embed(text):
        vec = np.zeros(dim)
        for char in text.lower():
            vec[ord(char) % dim] += 1
        norm = np.linalg.norm(vec)
        return (vec / norm).astype('float32') if norm > 0 else vec.astype('float32')

    embeddings = np.array([embed(doc['text']) for doc in knowledge_base])
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index, embed  # return embed fn so we can reuse it

faiss_index, embed_fn = build_faiss_index(KNOWLEDGE_BASE)
print(f'FAISS index built: {faiss_index.ntotal} documents')


def retrieve(query, top_k=3):
    """Retrieve top-k relevant documents for a query."""
    query_vec = embed_fn(query).reshape(1, -1)
    distances, indices = faiss_index.search(query_vec, top_k)
    return [KNOWLEDGE_BASE[i] for i in indices[0] if i < len(KNOWLEDGE_BASE)]


def generate_referral_report(mc_result, patient_id='UNKNOWN'):
    """
    RAG-assisted referral report generation.
    Retrieves relevant guidelines then calls Claude to write the report.
    """
    # Retrieve relevant clinical context
    query     = f"DR grade {mc_result['grade']} {mc_result['grade_label']} referral treatment"
    docs      = retrieve(query, top_k=3)
    context   = '\n\n'.join([f"[{d['source']}]\n{d['text']}" for d in docs])

    # Call Claude to generate structured report
    client = anthropic.Anthropic()  # needs ANTHROPIC_API_KEY in environment
    prompt = f"""You are a clinical AI assistant generating a diabetic retinopathy screening report.
Base recommendations ONLY on the provided guidelines. Do not invent medical advice.

VISION MODEL OUTPUT:
- Patient ID: {patient_id}
- DR Grade: {mc_result['grade']} ({mc_result['grade_label']})
- Confidence: {mc_result['confidence']:.2%}
- Uncertainty (variance): {mc_result['variance']:.4f}
- Referral triggered by: {'high uncertainty' if mc_result['variance'] > UNCERTAINTY_THRESHOLD else 'grade severity'}

RETRIEVED CLINICAL GUIDELINES:
{context}

Write a structured clinical report with these sections:
1. DIAGNOSIS SUMMARY
2. SEVERITY ASSESSMENT  
3. REFERRAL RECOMMENDATION (include urgency)
4. RECOMMENDED NEXT STEPS
5. UNCERTAINTY NOTE (explain model confidence to the clinician)

Be concise and clinical."""

    response = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=600,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response.content[0].text

print('RAG pipeline defined.')

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def run_dr_agent(image: Image.Image, patient_id='UNKNOWN', n_passes=30, verbose=True):
    """
    Full DR Screening Agent.

    State:  retinal image + model confidence score
    Actions: DIAGNOSE or REFER
    Policy:  high uncertainty OR high grade → REFER
             otherwise → DIAGNOSE
    """
    if verbose:
        print(f'\n{"="*50}')
        print(f'DR AGENT — Patient: {patient_id}')
        print(f'{"="*50}')

    # ── Step 1: Validate image ─────────────────────────
    if verbose: print('\n[1] Validating image quality...')
    quality = check_image_quality(image)
    if not quality['valid']:
        if verbose: print(f'    ✗ Rejected: {quality["reason"]}')
        return {'status': 'REJECTED', 'reason': quality['reason']}
    if verbose: print('    ✓ Image OK')

    # ── Step 2: Vision model + uncertainty ─────────────
    if verbose: print(f'\n[2] Running vision model ({n_passes} MC passes)...')
    img_tensor = val_transform(image).unsqueeze(0)
    mc_result  = mc_dropout_predict(model, img_tensor, n_passes=n_passes)

    if verbose:
        print(f'    Grade:      {mc_result["grade"]} — {mc_result["grade_label"]}')
        print(f'    Confidence: {mc_result["confidence"]:.2%}')
        print(f'    Variance:   {mc_result["variance"]:.4f}')

    # ── Step 3: Decision policy ────────────────────────
    high_uncertainty = mc_result['variance'] > UNCERTAINTY_THRESHOLD
    high_severity    = mc_result['grade'] >= 2
    needs_referral   = high_uncertainty or high_severity

    if verbose:
        print(f'\n[3] Decision policy:')
        print(f'    High uncertainty: {high_uncertainty}')
        print(f'    High severity:    {high_severity}')
        print(f'    → Action: {"REFER" if needs_referral else "DIAGNOSE"}')

    # ── Step 4: Generate report ────────────────────────
    if needs_referral:
        if verbose: print('\n[4] Generating RAG referral report...')
        report = generate_referral_report(mc_result, patient_id)
        action = 'REFER'
    else:
        if verbose: print('\n[4] Generating automated diagnosis...')
        report = (
            f'AUTOMATED DIAGNOSIS\n'
            f'Patient: {patient_id}\n'
            f'DR Grade: {mc_result["grade"]} ({mc_result["grade_label"]})\n'
            f'Confidence: {mc_result["confidence"]:.2%}\n'
            f'Recommendation: Annual follow-up. No referral required.'
        )
        action = 'DIAGNOSE'

    if verbose:
        print(f'\n{"─"*50}')
        print(report)

    return {
        'status':    'PROCESSED',
        'action':    action,
        'mc_result': mc_result,
        'report':    report
    }

print('Agent loop defined.')

In [ ]:
# Evaluation

# Load your trained checkpoint
model.load_state_dict(torch.load('checkpoints/dr_best.pth', map_location=DEVICE))

# Grab one image from val set
sample     = val_data[0]
image      = sample['image'] if isinstance(sample, dict) else aptos[val_data.indices[0]]['image']
image      = image.convert('RGB')

# Run the agent
result = run_dr_agent(image, patient_id='PT-001', n_passes=30)